# Chapter 19. Combining Datasets: merge and join

## 19.1 Relational Algebra

Database에서 사용되는 관계대수(Relational Algebra) 개념을 Pandas가 구현한 것이 pd.merge()입니다.

관계대수의 기본 연산들:

* Join  
* Projection (특정 렬 선택)  
* Selection (특정 행 선택)  
* Union, Intersection, Difference 등

## 19.2 Categories of Joins

### 19.2.1 One-to-One Join

두 DataFrame의 키 렬이 모두 유일한 값으로 구성된 경우

In [2]:
import numpy as np
import pandas as pd

df1 = pd.DataFrame({'employee': ['Bob', 'Jake', 'Lisa', 'Sue'],
                    'group': ['Accounting', 'Engineering', 'Engineering', 'HR']})
df2 = pd.DataFrame({'employee': ['Lisa', 'Bob', 'Jake', 'Sue'],
                    'hire_date': [2004, 2008, 2012, 2014]})

print("df1:")
print(df1)
print("\ndf2")
print(df2)

df1:
  employee        group
0      Bob   Accounting
1     Jake  Engineering
2     Lisa  Engineering
3      Sue           HR

df2
  employee  hire_date
0     Lisa       2004
1      Bob       2008
2     Jake       2012
3      Sue       2014


In [3]:
# 병합 (자동으로 공통 렬 'employee'를 키로 사용)
df3 = pd.merge(df1, df2)
print("\nresult of merge:")
print(df3)


result of merge:
  employee        group  hire_date
0      Bob   Accounting       2008
1     Jake  Engineering       2012
2     Lisa  Engineering       2004
3      Sue           HR       2014


### 19.2.2 Many-to-One Join

키 렬에 중복 값이 있는 경우

In [4]:
df4 = pd.DataFrame({'group': ['Accounting', 'Engineering', 'HR'],
                    'supervisor': ['Carly', 'Guido', 'Steve']})
print("df4:")
print(df4)

print("\nmerge of df3 and df4")
print(pd.merge(df3, df4))

df4:
         group supervisor
0   Accounting      Carly
1  Engineering      Guido
2           HR      Steve

merge of df3 and df4
  employee        group  hire_date supervisor
0      Bob   Accounting       2008      Carly
1     Jake  Engineering       2012      Guido
2     Lisa  Engineering       2004      Guido
3      Sue           HR       2014      Steve


### 19.2.3 Many-to-Many Join

두 DataFrame 모두 키 렬에 중복 값이 있는 경우

In [5]:
df5 = pd.DataFrame({'group': ['Accounting', 'Accounting',
                              'Engineering', 'Engineering', 'HR', 'HR'],
                    'skills': ['math', 'spreadsheets',
                               'software', 'math', 'spreadsheets', 'organization']})
print("df5:")
print(df5)

print("\nmerge of df1 and df5:")
print(pd.merge(df1, df5))

df5:
         group        skills
0   Accounting          math
1   Accounting  spreadsheets
2  Engineering      software
3  Engineering          math
4           HR  spreadsheets
5           HR  organization

merge of df1 and df5:
  employee        group        skills
0      Bob   Accounting          math
1      Bob   Accounting  spreadsheets
2     Jake  Engineering      software
3     Jake  Engineering          math
4     Lisa  Engineering      software
5     Lisa  Engineering          math
6      Sue           HR  spreadsheets
7      Sue           HR  organization


## 19.3 Specification of the Merge Key

### 19.3.1 The on Keyword

명시적으로 키로 사용할 렬을 지정할 때 사용합니다.

In [6]:
# 두 DataFrame에 동일한 렬 이름이 있을 때
print(pd.merge(df1, df2, on='employee'))

  employee        group  hire_date
0      Bob   Accounting       2008
1     Jake  Engineering       2012
2     Lisa  Engineering       2004
3      Sue           HR       2014


### 19.3.2 The left_on and right_on Keywords

량쪽 DataFrame의 키 렬 이름이 다를 때 사용합니다.

In [7]:
df3 = pd.DataFrame({'name': ['Bob', 'Jake', 'Lisa', 'Sue'],
                    'salary': [70000, 80000, 120000, 90000]})
print("df3:")
print(df3)

# df1의 'employee'와 df3의 'name'을 키로 련결
print("\nmerge result:")
print(pd.merge(df1, df3, left_on='employee', right_on='name'))

df3:
   name  salary
0   Bob   70000
1  Jake   80000
2  Lisa  120000
3   Sue   90000

merge result:
  employee        group  name  salary
0      Bob   Accounting   Bob   70000
1     Jake  Engineering  Jake   80000
2     Lisa  Engineering  Lisa  120000
3      Sue           HR   Sue   90000


💡 name 렬이 중복되므로 필요 없으면 drop()으로 제거할 수 있습니다.

In [8]:
print(pd.merge(df1, df3, left_on='employee', right_on='name').drop('name', axis=1))

  employee        group  salary
0      Bob   Accounting   70000
1     Jake  Engineering   80000
2     Lisa  Engineering  120000
3      Sue           HR   90000


### 19.3.3 The left_index and right_index Keywords

인덱스를 키로 사용하여 병합할 때 사용합니다.

In [9]:
df1a = df1.set_index('employee')
df2a = df2.set_index('employee')

print("df1a (index = employee):")
print(df1a)
print("\ndf2a (index = employee):")
print(df2a)

# index를 키로 병합
print("\nIndex 기준 병합:")
print(pd.merge(df1a, df2a, left_index=True, right_index=True))

df1a (index = employee):
                group
employee             
Bob        Accounting
Jake      Engineering
Lisa      Engineering
Sue                HR

df2a (index = employee):
          hire_date
employee           
Lisa           2004
Bob            2008
Jake           2012
Sue            2014

Index 기준 병합:
                group  hire_date
employee                        
Bob        Accounting       2008
Jake      Engineering       2012
Lisa      Engineering       2004
Sue                HR       2014


더 간단한 방법: join() method 사용

In [10]:
print(df1a.join(df2a))

                group  hire_date
employee                        
Bob        Accounting       2008
Jake      Engineering       2012
Lisa      Engineering       2004
Sue                HR       2014


## 19.4 Specifying Set Arithmetic for Joins

병합할 때 어떤 집합 연산을 사용할지 지정할 수 있습니다.

| how 값 | 의미 | 설명 |
|--------|------|------|
| `'inner'` | 교집합 | 량쪽 모두에 있는 키만 포함 (기본값) |
| `'outer'` | 합집합 | 모든 키를 포함, 없는 값은 NaN |
| `'left'` | 왼쪽 기준 | 왼쪽 DataFrame의 모든 키 포함 |
| `'right'` | 오른쪽 기준 | 오른쪽 DataFrame의 모든 키 포함 |

In [11]:
df6 = pd.DataFrame({'name': ['Peter', 'Paul', 'Mary'],
                    'food': ['fish', 'beans', 'bread']})
df7 = pd.DataFrame({'name': ['Mary', 'Joseph'],
                    'drink': ['wine', 'beer']})

print("df6:")
print(df6)
print("df7:")
print(df7)

df6:
    name   food
0  Peter   fish
1   Paul  beans
2   Mary  bread
df7:
     name drink
0    Mary  wine
1  Joseph  beer


In [12]:
# inner join (기본값)
print("\ninner join:")
print(pd.merge(df6, df7, how='inner'))

# outer join
print("\nouter join:")
print(pd.merge(df6, df7, how='outer'))

# left join
print("\nleft join:")
print(pd.merge(df6, df7, how='left'))

# right join
print("\nright join:")
print(pd.merge(df6, df7, how='right'))


inner join:
   name   food drink
0  Mary  bread  wine

outer join:
     name   food drink
0  Joseph    NaN  beer
1    Mary  bread  wine
2    Paul  beans   NaN
3   Peter   fish   NaN

left join:
    name   food drink
0  Peter   fish   NaN
1   Paul  beans   NaN
2   Mary  bread  wine

right join:
     name   food drink
0    Mary  bread  wine
1  Joseph    NaN  beer


## 19.5 Overlapping Column Names: The suffixes Keyword

병합할 때 두 DataFrame에 같은 이름의 렬이 있으면, 기본적으로 _x, _y가 붙습니다.

In [15]:
df8 = pd.DataFrame({'name': ['Bob', 'Jake', 'Lisa', 'Sue'],
                    'rank': [1, 2, 3, 4]})
df9 = pd.DataFrame({'name': ['Bob', 'Jake', 'Lisa', 'Sue'],
                    'rank': [1, 2, 3, 4]})

print(pd.merge(df8, df9, on='name'))

   name  rank_x  rank_y
0   Bob       1       1
1  Jake       2       2
2  Lisa       3       3
3   Sue       4       4


suffixes 키워드로 접미사를 변경할 수 있습니다.

In [16]:
print(pd.merge(df8, df9, on='name', suffixes=['_left', '_right']))

   name  rank_left  rank_right
0   Bob          1           1
1  Jake          2           2
2  Lisa          3           3
3   Sue          4           4
